# 🎙️ Speech Intelligence Engine — Google Colab GPU Trainer

This notebook offloads heavy **Speech Foundation Model** training, SSL feature extraction (WavLM / HuBERT), and Speech-LLM adapter fine-tuning to Google Colab GPUs (T4 / L4 / A100), persisting all checkpoints & cell execution telemetry directly to GitHub & 5TB **Google Drive** (`rakshithgoud453@gmail.com`).

## Step 1: Mount 5TB Google Drive & Initialize GitHub Telemetry Logger 📄

In [ ]:
import os
import sys
import time
import json
import traceback
from google.colab import drive

# Check available GPU resources
!nvidia-smi

# Mount 5TB Google Drive
drive.mount('/content/drive')

DRIVE_BASE_PATH = '/content/drive/MyDrive/speech_intelligence_engine'
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, 'checkpoints')
DATASET_DIR = os.path.join(DRIVE_BASE_PATH, 'datasets')
OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, 'outputs')
LOGS_DIR = '/content/speech_intelligence_engine/logs'

for path in [DRIVE_BASE_PATH, CHECKPOINT_DIR, DATASET_DIR, OUTPUT_DIR, LOGS_DIR]:
    os.makedirs(path, exist_ok=True)

def log_step(step_id: str, status: str, details: str = ""):
    payload = {
        "timestamp": time.ctime(),
        "step": step_id,
        "status": status,
        "details": details
    }
    step_file = os.path.join(LOGS_DIR, f"{step_id}_status.json")
    with open(step_file, "w") as f:
        json.dump(payload, f, indent=2)
    
    master_log = os.path.join(LOGS_DIR, "master_execution_log.txt")
    with open(master_log, "a") as f:
        f.write(f"[{time.ctime()}] [{step_id}] [{status}] {details[:300].strip()}\n")
    
    print(f"📄 Logged [{step_id}] -> [{status}]")
    if os.path.exists('/content/speech_intelligence_engine/.git'):
        os.system("cd /content/speech_intelligence_engine && git add logs/ && git commit -m 'telemetry: " + step_id + " " + status + "' && git push origin master")
        print(f"🚀 Auto-synced [{step_id}] telemetry to GitHub repository!")

log_step("step_1", "SUCCESS", "Mounted Google Drive & Initialized GitHub Telemetry")

## Step 2: Install ML & Audio Dependencies (Colab & NumPy 1.x Compatible)

In [ ]:
import traceback

try:
    print("Installing audio & ML dependencies with NumPy 1.x compatibility...")
    !pip install -q "numpy<2.0.0" transformers datasets peft accelerate bitsandbytes librosa soundfile wandb pyannote.audio faster-whisper hdbscan
    log_step("step_2", "SUCCESS", "Installed transformers, numpy<2.0.0, peft, librosa, hdbscan, pyannote, faster-whisper")
except Exception as e:
    err = traceback.format_exc()
    log_step("step_2", "FAILED", err)
    raise e

## Step 3: Clone / Sync Repository Code & Set Git Credentials

In [ ]:
import os
import traceback

try:
    REPO_URL = "https://github.com/rakshithgoud453/speech_intelligence_engine.git"
    WORKSPACE_DIR = "/content/speech_intelligence_engine"

    %cd /content
    if os.path.exists(WORKSPACE_DIR):
        %cd {WORKSPACE_DIR}
        !git pull
    else:
        !git clone {REPO_URL} {WORKSPACE_DIR}
        %cd {WORKSPACE_DIR}

    # Configure Git in Colab workspace for automated telemetry pushes
    !git config user.name "rakshithgoud453"
    !git config user.email "rakshithgoud453@gmail.com"

    print(f"✓ Currently operating in: {os.getcwd()}")
    log_step("step_3", "SUCCESS", f"Repo synced at {os.getcwd()}")
except Exception as e:
    err = traceback.format_exc()
    log_step("step_3", "FAILED", err)
    raise e

## Step 4: SSL Layer Representation Extraction (WavLM / HuBERT)

In [ ]:
import torch
import torchaudio
import traceback

try:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Torchaudio native WavLM pipeline
    bundle = torchaudio.pipelines.WAVLM_BASE
    ssl_model = bundle.get_model().to(device)
    ssl_model.eval()

    def extract_ssl_representations(audio_path):
        waveform, sample_rate = torchaudio.load(audio_path)
        if sample_rate != bundle.sample_rate:
            resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=bundle.sample_rate)
            waveform = resampler(waveform)
        
        tensor_wav = waveform.to(device)
        if tensor_wav.ndim == 1:
            tensor_wav = tensor_wav.unsqueeze(0)
        if tensor_wav.shape[1] < 1600:
            tensor_wav = torch.nn.functional.pad(tensor_wav, (0, 1600 - tensor_wav.shape[1]))
            
        with torch.no_grad():
            hidden_states, _ = ssl_model.extract_features(tensor_wav)
        
        prosodic_rep = torch.stack(hidden_states[4:9]).mean(dim=0)[0]
        semantic_rep = torch.stack(hidden_states[9:12]).mean(dim=0)[0]
        
        return {
            "prosodic_embedding": prosodic_rep.cpu().numpy(),
            "semantic_embedding": semantic_rep.cpu().numpy()
        }

    print("✓ SSL Feature Extraction Pipeline Loaded Successfully.")
    log_step("step_4", "SUCCESS", f"Loaded torchaudio WAVLM_BASE on {device}")
except Exception as e:
    err = traceback.format_exc()
    log_step("step_4", "FAILED", err)
    raise e

## Step 5: Checkpoint Saver Utility

In [ ]:
import os
import time
import torch
import traceback

try:
    DRIVE_BASE_PATH = '/content/drive/MyDrive/speech_intelligence_engine'
    CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, 'checkpoints')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    def save_checkpoint(model, step, metrics):
        checkpoint_name = f"speech_engine_step_{step}_{int(time.time())}.pt"
        save_path = os.path.join(CHECKPOINT_DIR, checkpoint_name)
        
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),
            'metrics': metrics
        }, save_path)
        
        print(f"✓ Saved Checkpoint to Google Drive (5TB): {save_path}")

    print(f"✓ Checkpoints directory configured: {CHECKPOINT_DIR}")
    log_step("step_5", "SUCCESS", f"Checkpoint directory ready at {CHECKPOINT_DIR}")
except Exception as e:
    err = traceback.format_exc()
    log_step("step_5", "FAILED", err)
    raise e

## Step 6: Batch Audio Ingestion & Model Checkpoint Execution Loop 🚀

In [ ]:
import sys
import glob
import json
import os
import time
import traceback
import soundfile as sf
import numpy as np
import torch

try:
    DRIVE_BASE_PATH = '/content/drive/MyDrive/speech_intelligence_engine'
    CHECKPOINT_DIR = os.path.join(DRIVE_BASE_PATH, 'checkpoints')
    DATASET_DIR = os.path.join(DRIVE_BASE_PATH, 'datasets')
    OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, 'outputs')
    for p in [CHECKPOINT_DIR, DATASET_DIR, OUTPUT_DIR]:
        os.makedirs(p, exist_ok=True)

    WORKSPACE_DIR = "/content/speech_intelligence_engine"
    for p in [WORKSPACE_DIR, os.path.dirname(WORKSPACE_DIR)]:
        if p not in sys.path:
            sys.path.insert(0, p)

    try:
        from speech_intelligence_engine import SpeechLearningEngine
    except ImportError:
        from engine import SpeechLearningEngine

    try:
        from speech_intelligence_engine.representation.projector import SpeechLLMProjector
    except ImportError:
        from representation.projector import SpeechLLMProjector

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Initializing Engine & Projector on device: {device}...")

    engine = SpeechLearningEngine(model_type="wavlm", replay_capacity=1000)
    projector = SpeechLLMProjector(input_dim=768, hidden_dim=2048, output_dim=4096).to(device)

    audio_files = glob.glob(os.path.join(DATASET_DIR, "*.wav")) + glob.glob(os.path.join(DATASET_DIR, "*.mp3"))

    if not audio_files:
        print(f"No user audio files found in Google Drive datasets folder. Generating benchmark audio sample...")
        sample_wav = os.path.join(DATASET_DIR, "sample_speech_benchmark.wav")
        t = np.linspace(0, 3.0, int(16000 * 3.0), endpoint=False)
        sig = (0.5 * np.sin(2 * np.pi * 180 * t) + 0.2 * np.sin(2 * np.pi * 360 * t)).astype(np.float32)
        sf.write(sample_wav, sig, 16000)
        audio_files = [sample_wav]

    for i, audio_path in enumerate(audio_files, 1):
        print(f"\n[Processing {i}/{len(audio_files)}]: {audio_path}")
        res = engine.assimilate(audio_path, recording_id=os.path.basename(audio_path))
        
        metrics = {
            "episodes": res.num_episodes,
            "voiced_segments": res.num_segments,
            "novelty_rate": res.novelty_rate,
            "duration_sec": res.duration_sec
        }
        save_checkpoint(projector, step=i, metrics=metrics)

    status_payload = {
        "status": "COMPLETED",
        "total_episodes_learned": engine.get_state().total_episodes,
        "prototypes_discovered": engine.get_state().total_prototypes,
        "checkpoint_directory": CHECKPOINT_DIR,
        "last_updated": time.ctime()
    }

    status_file = os.path.join(OUTPUT_DIR, "status.json")
    with open(status_file, "w") as f:
        json.dump(status_payload, f, indent=2)

    print(f"\n✨ Batch processing complete!")
    print(f"📁 Checkpoint saved to Google Drive: {CHECKPOINT_DIR}")
    print(f"📄 Status summary written to: {status_file}")
    log_step("step_6", "SUCCESS", f"Batch completed. Episodes: {engine.get_state().total_episodes}, Prototypes: {engine.get_state().total_prototypes}")

except Exception as e:
    err_trace = traceback.format_exc()
    log_step("step_6", "FAILED", err_trace)
    print(f"\n❌ EXECUTION ERROR IN STEP 6:\n{err_trace}")
    raise e